# Spotify User Behavior — Data Cleaning & Quality Assessment

**Purpose:** This notebook loads the raw Spotify user behavior dataset, assesses its data quality issues, applies documented cleaning decisions, and exports a validated dataset for downstream analysis in Excel and Power BI.

**Author:** Akhil T V

**Date:** 11/06/2026

**Workflow:** Raw CSV → Quality Assessment → Cleaning → Validation → Export to `cleaned_spotify_dataset.csv`


## Step 1 — Setup & Load

In [71]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

# Load raw dataset
df = pd.read_csv('data/raw_spotify_dataset.csv')
print(f"Loaded dataset with {df.shape[0]:,} rows and {df.shape[1]} columns")


Loaded dataset with 50,050 rows and 18 columns


## Step 2 — Initial Inspection

In [72]:
df.shape

(50050, 18)

In [73]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50050 entries, 0 to 50049
Data columns (total 18 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   user_id                         50050 non-null  int64  
 1   country                         50050 non-null  object 
 2   age                             48299 non-null  float64
 3   signup_date                     50050 non-null  object 
 4   subscription_type               50050 non-null  object 
 5   subscription_status             50050 non-null  object 
 6   months_inactive                 50050 non-null  int64  
 7   inactive_3_months_flag          50050 non-null  int64  
 8   ad_interaction                  50050 non-null  object 
 9   ad_conversion_to_subscription   50050 non-null  object 
 10  music_suggestion_rating_1_to_5  45678 non-null  float64
 11  avg_listening_hours_per_week    50050 non-null  float64
 12  favorite_genre                  

In [74]:
df.head()

,user_id,country,age,signup_date,subscription_type,subscription_status,months_inactive,inactive_3_months_flag,ad_interaction,ad_conversion_to_subscription,music_suggestion_rating_1_to_5,avg_listening_hours_per_week,favorite_genre,most_liked_feature,desired_future_feature,primary_device,playlists_created,avg_skips_per_day
0,17397,USA,38.0,2020-07-18,Free,Inactive,3,1,No,No,5.0,8.37,Jazz,AI DJ,NaN,Mobile,7,7
1,21810,Canada,29.0,2025-01-01,Free,Active,0,0,No,No,3.0,3.82,K-Pop,AI DJ,NaN,Car System,3,12
2,5717,Indonesia,24.0,2023-10-21,Premium Individual,Active,1,0,Yes,No,3.0,17.10,Bollywood,Podcasts,Concert Alerts,Tablet,9,11
3,18847,France,52.0,2025-07-11,Premium Duo,Active,0,0,Yes,No,5.0,6.67,Electronic,Playlists,NaN,Tablet,10,11
4,10745,Spain,36.0,2022-04-27,Free,Active,1,0,No,No,4.0,11.26,R&B,Playlists,Lyrics Translation,Smart Speaker,7,8


In [75]:
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
user_id,50050.0,NaN,NaN,NaN,25006.855584,14438.876837,1.0,12503.25,25004.5,37509.75,50020.0
country,50050,15,India,4269,NaN,NaN,NaN,NaN,NaN,NaN,NaN
age,48299.0,NaN,NaN,NaN,37.993892,13.035731,2.0,27.0,38.0,49.0,120.0
signup_date,50050,3002,2026-01-18,33,NaN,NaN,NaN,NaN,NaN,NaN,NaN
subscription_type,50050,5,Free,22549,NaN,NaN,NaN,NaN,NaN,NaN,NaN
subscription_status,50050,2,Active,42154,NaN,NaN,NaN,NaN,NaN,NaN,NaN
months_inactive,50050.0,NaN,NaN,NaN,1.532547,1.951691,0.0,0.0,1.0,2.0,18.0
inactive_3_months_flag,50050.0,NaN,NaN,NaN,0.222378,0.415848,0.0,0.0,0.0,0.0,1.0
ad_interaction,50050,2,No,32538,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ad_conversion_to_subscription,50050,2,No,45689,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**Initial observations:**
- The dataset contains 50,050 records and 18 columns, covering user demographics, subscription behavior, listening activity, engagement, and preference-related information.
- Initial inspection identifies missing values in age, favorite_genre, music_suggestion_rating_1_to_5, and desired_future_feature, while the remaining columns are complete.
- The descriptive statistics reveal several data quality issues, including implausible age values, negative listening hours, negative playlist counts, and inconsistent categorical values, indicating that data cleaning is necessary before analysis.

## Step 3 — Data Quality Assessment

Before making any changes, we assess the dataset to understand the scope and nature of data quality issues. This assessment is done in full before any fix is applied, so decisions in later steps are based on a complete picture rather than issue-by-issue reactions.


### 3.1 Missing Values

In [76]:
missing_summary = pd.DataFrame({
    'missing_count': df.isnull().sum(),
    'missing_pct': (df.isnull().sum() / len(df) * 100).round(2)
})
missing_summary = missing_summary[missing_summary['missing_count'] > 0].sort_values('missing_count', ascending=False)
missing_summary

,missing_count,missing_pct
desired_future_feature,20021,40.00
music_suggestion_rating_1_to_5,4372,8.74
favorite_genre,2504,5.00
age,1751,3.50


### 3.2 Duplicate Records

In [77]:
exact_duplicates = df.duplicated().sum()
duplicate_user_ids = df['user_id'].duplicated().sum()

print(f"Fully identical duplicate rows: {exact_duplicates}")
print(f"Duplicate user_id values: {duplicate_user_ids}")

Fully identical duplicate rows: 30
Duplicate user_id values: 30


In [78]:
# Check for near-duplicates: same profile data but a DIFFERENT user_id.
# These won't be caught by checking user_id alone, since a duplicate/bot
# account would typically be assigned a new ID rather than reusing one.
profile_cols = [c for c in df.columns if c != 'user_id']
near_dupe_mask = df.duplicated(subset=profile_cols, keep=False)
print(f"Rows sharing identical profile data under a different user_id: {near_dupe_mask.sum()}")
df[near_dupe_mask].sort_values(profile_cols[0]).head(10)

Rows sharing identical profile data under a different user_id: 100


,user_id,country,age,signup_date,subscription_type,subscription_status,months_inactive,inactive_3_months_flag,ad_interaction,ad_conversion_to_subscription,music_suggestion_rating_1_to_5,avg_listening_hours_per_week,favorite_genre,most_liked_feature,desired_future_feature,primary_device,playlists_created,avg_skips_per_day
18521,50006,Australia,59.0,2020-04-27,Free,Active,0,0,Yes,Yes,4.0,9.56,Rock,Discover Weekly,Mood-based Auto Playlists,Tablet,5,8
17568,21219,Australia,39.0,2023-07-25,Student,Active,0,0,No,No,4.0,10.62,Bollywood,Radio,Mood-based Auto Playlists,Smart Speaker,5,11
14512,37886,Australia,18.0,2023-06-15,Premium Family,Active,1,0,No,No,1.0,3.66,Country,Podcasts,Concert Alerts,Tablet,6,8
46040,2242,Australia,55.0,2023-08-21,Free,Active,1,0,Yes,No,NaN,3.58,Pop,Daily Mix,Social Listening,Smart Speaker,13,12
45460,2242,Australia,55.0,2023-08-21,Free,Active,1,0,Yes,No,NaN,3.58,Pop,Daily Mix,Social Listening,Smart Speaker,13,12
33520,37886,Australia,18.0,2023-06-15,Premium Family,Active,1,0,No,No,1.0,3.66,Country,Podcasts,Concert Alerts,Tablet,6,8
34209,21219,Australia,39.0,2023-07-25,Student,Active,0,0,No,No,4.0,10.62,Bollywood,Radio,Mood-based Auto Playlists,Smart Speaker,5,11
21722,34672,Australia,59.0,2020-04-27,Free,Active,0,0,Yes,Yes,4.0,9.56,Rock,Discover Weekly,Mood-based Auto Playlists,Tablet,5,8
11801,4797,Brazil,33.0,2022-12-26,Free,Active,0,0,Yes,No,3.0,11.28,Electronic,Daily Mix,Mood-based Auto Playlists,Desktop,10,7
24488,5635,Brazil,57.0,2023-10-15,Student,Active,1,0,Yes,Yes,4.0,13.11,Classical,Podcasts,Lyrics Translation,SMART SPEAKER,9,11


### 3.3 Invalid / Out-of-Range Values

In [79]:
print("Age range:", df['age'].min(), "to", df['age'].max())
print("Users with implausible age (<10 or >100):", ((df['age'] < 10) | (df['age'] > 100)).sum())
print()
print("Negative avg_listening_hours_per_week:", (df['avg_listening_hours_per_week'] < 0).sum())
print("Negative playlists_created:", (df['playlists_created'] < 0).sum())
print()
print("Extreme avg_listening_hours_per_week (>100 hrs/week):", (df['avg_listening_hours_per_week'] > 100).sum())
print("Extreme playlists_created (>150):", (df['playlists_created'] > 150).sum())

Age range: 2.0 to 120.0
Users with implausible age (<10 or >100): 25

Negative avg_listening_hours_per_week: 15
Negative playlists_created: 10

Extreme avg_listening_hours_per_week (>100 hrs/week): 12
Extreme playlists_created (>150): 10


### 3.4 Categorical Text Consistency

In [80]:
print("Unique country values:")
print(sorted(df['country'].dropna().unique()))

Unique country values:
['Australia', 'Brazil', 'Canada', 'France', 'Germany', 'India', 'Indonesia', 'Italy', 'Mexico', 'Spain', 'U.K.', 'UK', 'USA', 'United Kingdom', 'uk']


In [81]:
print("Unique primary_device values:")
print(sorted(df['primary_device'].dropna().unique()))

Unique primary_device values:
['CAR SYSTEM', 'Car System', 'DESKTOP', 'Desktop', 'MOBILE', 'Mobile', 'SMART SPEAKER', 'Smart Speaker', 'TABLET', 'Tablet', 'car system', 'desktop', 'mobile', 'smart speaker', 'tablet']


### 3.5 Timeline / Logical Consistency

In [82]:
df['signup_date'] = pd.to_datetime(df['signup_date'])
max_reference_date = df['signup_date'].max()
print("Latest signup date in dataset:", max_reference_date)

# Flag signups that fall suspiciously close to or after the dataset's own max date
future_signups = df[df['signup_date'] > pd.Timestamp('2026-07-01')]
print(f"Signups dated after July 2026 (likely erroneous): {len(future_signups)}")

Latest signup date in dataset: 2027-02-08 00:00:00
Signups dated after July 2026 (likely erroneous): 17


### Data Quality Assessment Summary

| Column | Issue | Count | Severity |
|---|---|---|---|
| age | Missing | 1,751 | Medium |
| age | Invalid (Age <10 or >100) | 25 | High |
| favorite_genre | Missing | 2,504 | Low |
| music_suggestion_rating_1_to_5 | Missing | 4,372 | Medium |
| desired_future_feature | Missing | 20,021 | Low |
| country | Inconsistent spelling/casing | 4,163 | Medium |
| primary_device | Inconsistent casing | 4,030 | Low |
| avg_listening_hours_per_week | Negative values | 15 | High |
| playlists_created | Negative values | 10 | High |
| (various) | Exact duplicate rows | 30 | High |
| (various) | Near-duplicate accounts | 40 records (20 duplicate profile pairs) | Medium |
| signup_date | Future-dated signups | 17 | Medium |
| avg_listening_hours_per_week / playlists_created | Extreme outliers (>100 hrs/week) | ~22 | Low |
| playlists_created | Extreme outliers (>150 playlists) | 10 | Low |


## Step 4 — Handle Duplicates

**Decision:** Exact duplicate rows are removed — they represent the same user record appearing twice, most likely from a duplicate export or ETL re-run, and provide no additional information.

Near-duplicate accounts (same profile attributes, different `user_id`) are **not deleted**, since they could represent legitimate re-signups after an account deletion. Instead, they are flagged with a new column for downstream review, preserving the data while noting the uncertainty.


In [83]:
before = len(df)
df = df.drop_duplicates()
after = len(df)
print(f"Removed {before - after} exact duplicate rows. New row count: {after:,}")

Removed 30 exact duplicate rows. New row count: 50,020


In [84]:
# Flag near-duplicate accounts (identical profile data, different user_id) without deleting them
profile_cols = [c for c in df.columns if c != 'user_id']
df['flag_duplicate_profile'] = df.duplicated(subset=profile_cols, keep=False)
print(f"Flagged {df['flag_duplicate_profile'].sum()} rows as duplicate profile data for review")

Flagged 40 rows as duplicate profile data for review


## Step 5 — Handle Missing Values

Each column is handled individually, since a single blanket strategy (e.g. "fill everything with the mean" or "drop every row with any null") is not appropriate — the right approach depends on what the column represents and why the value might be missing.


### 5.1 `age` — missing values

**Decision:** Leave as null (do not impute).

**Justification:**
- Age is a demographic attribute with no reliable information available elsewhere in the dataset to accurately estimate missing values.
- Imputing the mean or median would introduce artificial demographic information and could bias age-based segmentation and analysis.
- Therefore, missing age values are intentionally retained as null and excluded only from analyses that specifically require age.


In [85]:
# No action needed — age remains null where missing.
# It will be automatically excluded from age-based groupby operations later.
print(f"Rows with missing age: {df['age'].isnull().sum()}")

Rows with missing age: 1750


### 5.2 `favorite_genre` — missing values

**Decision:** Fill with `'Not Specified'`.

**Justification:**
- Favorite genre is a user preference rather than a measurable value, making it inappropriate to infer or predict missing responses.
- A missing value most likely indicates that the user never selected a preferred genre rather than the data being incorrect.
- Replacing missing values with "Not Specified" preserves all records while clearly distinguishing users who did not provide this information.


In [86]:
df['favorite_genre'] = df['favorite_genre'].fillna('Not Specified')
print(df['favorite_genre'].isnull().sum(), "missing values remaining")

0 missing values remaining


### 5.3 `music_suggestion_rating_1_to_5` — missing values

**Decision:** Leave as null; do not impute with mean/median.

**Justification:**
- Recommendation ratings are only available for users who actively interacted with the recommendation system, making the missing values informative rather than random.
- Filling missing ratings with an average or median would incorrectly imply user satisfaction where no feedback was actually provided.
- Therefore, missing ratings are intentionally preserved and an additional indicator column is created to identify users who never submitted a rating.


In [87]:
print(f"Rows with missing rating: {df['music_suggestion_rating_1_to_5'].isnull().sum()}")
# Optional: create a flag so downstream analysis can treat 'never rated' as its own segment
df['flag_no_rating_given'] = df['music_suggestion_rating_1_to_5'].isnull()

Rows with missing rating: 4371


### 5.4 `desired_future_feature` — missing values

**Decision:** Fill with `'No Response'`.

**Justification:**
- This field represents optional user feedback, so a high proportion of missing responses is expected and does not necessarily indicate poor data quality.
- Filling missing values with "No Response" preserves every record while accurately distinguishing users who chose not to provide feature suggestions.
- This approach also simplifies aggregation and visualization without introducing fabricated preferences.


In [88]:
df['desired_future_feature'] = df['desired_future_feature'].fillna('No Response')
print(df['desired_future_feature'].isnull().sum(), "missing values remaining")

0 missing values remaining


## Step 6 — Handle Invalid / Impossible Values

**Decision on age:** Values outside a plausible range (e.g. under 10 or over 100) are treated as data entry errors and set to null rather than guessed at or dropped entirely, consistent with how missing age is already handled.

**Decision on negative values:** Negative `avg_listening_hours_per_week` and `playlists_created` are physically impossible, so they are corrected to their absolute value — consistent with a likely sign/calculation error upstream rather than a meaningless entry.

**Decision on extreme outliers:** Very high but *possible* values (e.g. 100+ hrs/week listening) are not deleted or capped — they are flagged for review, since aggressive removal risks discarding real power users. Deleting them would understate genuine variation in the data.


In [89]:
# Invalid age values -> null
invalid_age_mask = (df['age'] < 10) | (df['age'] > 100)
print(f"Invalid age values found: {invalid_age_mask.sum()}")
df.loc[invalid_age_mask, 'age'] = np.nan

Invalid age values found: 25


In [90]:
# Negative values -> corrected to absolute value
neg_hours_mask = df['avg_listening_hours_per_week'] < 0
neg_playlists_mask = df['playlists_created'] < 0

print(f"Negative listening hours corrected: {neg_hours_mask.sum()}")
print(f"Negative playlist counts corrected: {neg_playlists_mask.sum()}")

df['avg_listening_hours_per_week'] = df['avg_listening_hours_per_week'].abs()
df['playlists_created'] = df['playlists_created'].abs()

Negative listening hours corrected: 15
Negative playlist counts corrected: 10


In [91]:
# Flag extreme outliers for downstream review rather than removing them
df['flag_outlier_listening_hours'] = df['avg_listening_hours_per_week'] > 100
df['flag_outlier_playlists'] = df['playlists_created'] > 150

print(f"Outlier listening hours flagged: {df['flag_outlier_listening_hours'].sum()}")
print(f"Outlier playlist counts flagged: {df['flag_outlier_playlists'].sum()}")

Outlier listening hours flagged: 12
Outlier playlist counts flagged: 10


## Step 7 — Standardize Categorical Text

**Decision:** Map all spelling/casing variants of the same country to a single canonical value, and standardize device names to consistent title case.

**Justification:**
- Standardizing country names and device labels ensures that identical categories are grouped together during analysis.
- Without standardization, variations such as UK, U.K., and United Kingdom would be treated as separate categories, leading to inaccurate counts and misleading visualizations.
- Applying consistent naming conventions improves reporting accuracy across Excel, Python, and Power BI.


In [92]:
country_mapping = {
    'usa': 'United States',
    'u.s.a': 'United States',
    'united states of america': 'United States',
    'united states': 'United States',
    'uk': 'United Kingdom',
    'u.k.': 'United Kingdom',
    'united kingdom': 'United Kingdom',
}

df['country'] = df['country'].str.strip().str.lower().map(country_mapping).fillna(df['country'])
print(sorted(df['country'].unique()))

['Australia', 'Brazil', 'Canada', 'France', 'Germany', 'India', 'Indonesia', 'Italy', 'Mexico', 'Spain', 'United Kingdom', 'United States']


In [93]:
df['primary_device'] = df['primary_device'].str.strip().str.title()
print(sorted(df['primary_device'].unique()))

['Car System', 'Desktop', 'Mobile', 'Smart Speaker', 'Tablet']


## Step 8 — Fix Timeline Inconsistencies

**Decision:** Signup dates falling after the dataset's effective reference date are flagged as erroneous and excluded from time-trend analysis, rather than silently corrected — since there's no reliable way to know what the "true" date should have been.

**Justification:**
- Records containing signup dates beyond the project's reference period are treated as invalid rather than automatically corrected.
- Since the true signup dates cannot be verified, modifying them would introduce assumptions that cannot be justified.
- Flagging these records preserves transparency while allowing analysts to exclude them from time-based reporting when necessary.

In [94]:
reference_date = pd.Timestamp('2026-07-01')
df['flag_invalid_signup_date'] = df['signup_date'] > reference_date
print(f"Rows flagged for invalid/future signup_date: {df['flag_invalid_signup_date'].sum()}")

Rows flagged for invalid/future signup_date: 17


## Step 9 — Final Validation

Re-running the same checks from Step 3 to confirm the cleaning steps worked as intended, and to produce a clear before/after comparison.


In [95]:
print("=== FINAL VALIDATION ===\n")

print("Remaining missing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print()

print("Remaining exact duplicate rows:", df.duplicated().sum())
print("Remaining negative listening hours:", (df['avg_listening_hours_per_week'] < 0).sum())
print("Remaining negative playlist counts:", (df['playlists_created'] < 0).sum())
print("Remaining implausible ages (excluding nulls):", ((df['age'] < 10) | (df['age'] > 100)).sum())
print()
print("Unique country values after standardization:", df['country'].nunique())
print("Unique device values after standardization:", df['primary_device'].nunique())
print()
print(f"Final dataset shape: {df.shape}")

=== FINAL VALIDATION ===

Remaining missing values:
age                               1775
music_suggestion_rating_1_to_5    4371
dtype: int64

Remaining exact duplicate rows: 0
Remaining negative listening hours: 0
Remaining negative playlist counts: 0
Remaining implausible ages (excluding nulls): 0

Unique country values after standardization: 12
Unique device values after standardization: 5

Final dataset shape: (50020, 23)


**Note on remaining nulls:** `age` and `music_suggestion_rating_1_to_5` still contain nulls by design (see Steps 5.1 and 5.3) — these were intentionally left as missing rather than imputed, since fabricating values would be less accurate than acknowledging the gap. All other fields were resolved.


## Step 10 — Feature Engineering: Helper Columns for Analysis

These derived columns match the exact bucketing logic used in the original project's Excel
workbook, so segment definitions stay consistent across the notebook, Excel, and Power BI.
Bins were confirmed against the original `spotify_dataset.xlsx` source file, not re-guessed.


In [96]:
# Age Group — confirmed against original workbook bins
age_bins = [15, 24, 34, 44, 54, 61]
age_labels = ['Gen Z', 'Young Adults', 'Mid-Career', 'Mature Adults', 'Seniors']
df['Age Group'] = pd.cut(df['age'], bins=age_bins, labels=age_labels)

print(df['Age Group'].value_counts(dropna=False))# Age Group — confirmed against original workbook bins
age_bins = [15, 24, 34, 44, 54, 61]
age_labels = ['Gen Z', 'Young Adults', 'Mid-Career', 'Mature Adults', 'Seniors']
df['Age Group'] = pd.cut(df['age'], bins=age_bins, labels=age_labels)

print(df['Age Group'].value_counts(dropna=False))

Age Group
Mature Adults    10893
Mid-Career       10705
Young Adults     10618
Gen Z             9688
Seniors           6341
NaN               1775
Name: count, dtype: int64
Age Group
Mature Adults    10893
Mid-Career       10705
Young Adults     10618
Gen Z             9688
Seniors           6341
NaN               1775
Name: count, dtype: int64


In [97]:
# Signup Month / Year — extracted directly from signup_date, matches original workbook
df['Month'] = df['signup_date'].dt.month
df['Year'] = df['signup_date'].dt.year

print(df[['signup_date', 'Month', 'Year']].head())

  signup_date  Month  Year
0  2020-07-18      7  2020
1  2025-01-01      1  2025
2  2023-10-21     10  2023
3  2025-07-11      7  2025
4  2022-04-27      4  2022


In [98]:
# Listening Hour Category — confirmed against original workbook bins:
# Low 0-5 | Medium 5.01-15 | High 15.01-24.64 | Very High 25.03+
def listening_category(hrs):
    if pd.isna(hrs):
        return np.nan
    elif hrs <= 5:
        return 'Low'
    elif hrs <= 15:
        return 'Medium'
    elif hrs <= 24.64:
        return 'High'
    else:
        return 'Very High'

df['Listening Hour Category'] = df['avg_listening_hours_per_week'].apply(listening_category)
print(df['Listening Hour Category'].value_counts(dropna=False))

Listening Hour Category
Medium       39561
Low           5345
High          5095
Very High       19
Name: count, dtype: int64


In [99]:
# Playlist Creator Category — confirmed against original workbook bins:
# None 0 | Casual 1-5 | Active 6-15 | Power User 16-23
# Note: values above 23 exist in this cleaned dataset (extreme outliers flagged in Step 6/
# flag_outlier_playlists); these are capped into 'Power User' rather than left unlabeled,
# since the original workbook's top bin had no defined upper limit for outliers.
def playlist_category(n):
    if pd.isna(n):
        return np.nan
    elif n == 0:
        return 'None'
    elif n <= 5:
        return 'Casual'
    elif n <= 15:
        return 'Active'
    else:
        return 'Power User'

df['Playlist Creator Category'] = df['playlists_created'].apply(playlist_category)
print(df['Playlist Creator Category'].value_counts(dropna=False))

Playlist Creator Category
Active        39947
Casual         9628
Power User      432
None             13
Name: count, dtype: int64


In [100]:
# Premium flag — any subscription_type other than 'Free'
df['is_premium'] = (df['subscription_type'] != 'Free').astype(int)
print(df['is_premium'].value_counts())

is_premium
1    27484
0    22536
Name: count, dtype: int64


In [101]:
# Churn flag — numeric version of subscription_status, used for correlation/statistical tests
df['is_churned'] = (df['subscription_status'] == 'Inactive').astype(int)
print(df['is_churned'].value_counts())

is_churned
0    42126
1     7894
Name: count, dtype: int64


## Step 11 — Export Cleaned Dataset

In [102]:
df.to_csv('data/cleaned_spotify_dataset.csv', index=False)
print("Exported to data/cleaned_spotify_dataset.csv")

Exported to data/cleaned_spotify_dataset.csv


### Summary

- **Starting row count:** 50,050
- **Final row count:** 50,020
- **Rows removed:** 30 exact duplicate records
- **Columns with intentional remaining nulls:** age, music_suggestion_rating_1_to_5
- **New flag columns added:** flag_duplicate_profile, flag_no_rating_given, flag_outlier_listening_hours, flag_outlier_playlists, flag_invalid_signup_date
- **Known limitations:** Near-duplicate profiles were flagged rather than removed because additional business context is required to determine whether they represent legitimate multiple accounts or duplicate records. Similarly, missing demographic and recommendation rating values were intentionally retained to avoid introducing artificial information into the dataset.


## Step 12 — Statistical Significance Testing

Several dashboard charts present differences between groups (e.g., churn rate by 
subscription type, average rating by genre). Before treating any of these as real 
findings, we test whether the observed differences are statistically significant 
or simply within the range of random variation.

- **Chi-square test** is used for categorical comparisons (e.g., churn rate across 
  subscription types or countries), since both the outcome and the grouping variable 
  are categorical.
- **One-way ANOVA** is used for comparing a numeric average (e.g., rating, listening 
  hours) across more than two groups.

A p-value below 0.05 is treated as the conventional threshold for statistical 
significance in this analysis.

In [103]:
import scipy.stats as stats

results = []

# Test 1: Churn rate by subscription_type
contingency_sub = pd.crosstab(df['subscription_type'], df['subscription_status'])
chi2_sub, p_sub, dof_sub, _ = stats.chi2_contingency(contingency_sub)
results.append(('Churn rate by subscription_type', 'Chi-square', p_sub))

# Test 2: Churn rate by country
contingency_country = pd.crosstab(df['country'], df['subscription_status'])
chi2_country, p_country, dof_country, _ = stats.chi2_contingency(contingency_country)
results.append(('Churn rate by country', 'Chi-square', p_country))

# Test 3: Avg recommendation rating by favorite_genre
genre_groups = [g['music_suggestion_rating_1_to_5'].dropna() 
                for _, g in df.groupby('favorite_genre', observed=True)]
f_genre, p_genre = stats.f_oneway(*genre_groups)
results.append(('Avg recommendation rating by genre', 'ANOVA', p_genre))

# Test 4: Avg listening hours by Age Group
age_groups = [g['avg_listening_hours_per_week'].dropna() 
              for _, g in df.groupby('Age Group', observed=True)]
f_age, p_age = stats.f_oneway(*age_groups)
results.append(('Avg listening hours by Age Group', 'ANOVA', p_age))

In [104]:
sig_results = pd.DataFrame(results, columns=['Comparison', 'Test Used', 'p_value'])
sig_results['Significant (p < 0.05)?'] = sig_results['p_value'] < 0.05
sig_results['Verdict'] = sig_results['Significant (p < 0.05)?'].map({
    True: 'Real pattern — safe to present as a finding',
    False: 'Not significant — likely random noise, should NOT be presented as a real difference'
})
sig_results

,Comparison,Test Used,p_value,Significant (p < 0.05)?,Verdict
0,Churn rate by subscription_type,Chi-square,0.559138,False,"Not significant — likely random noise, should ..."
1,Churn rate by country,Chi-square,0.506530,False,"Not significant — likely random noise, should ..."
2,Avg recommendation rating by genre,ANOVA,0.876735,False,"Not significant — likely random noise, should ..."
3,Avg listening hours by Age Group,ANOVA,0.166253,False,"Not significant — likely random noise, should ..."


### Interpretation

- The statistical tests indicate that none of the evaluated group comparisons are statistically significant at the 5% significance level (p < 0.05).
- Although visual differences appear across subscription types, countries, genres, and age groups, these differences are not strong enough to conclude that a true relationship exists within the population.
- These results should therefore be interpreted as descriptive summaries rather than statistically validated findings, and any business decisions should be supported by additional analysis or larger evidence beyond these comparisons.